# Part 2: GenAI Prompt Engineering with Flan-T5

## Scenario (continued)
Your startup's leadership is also interested in using AI to **automatically analyze customer reviews** — summarizing feedback, classifying sentiment, and extracting key information. Instead of training a model from scratch, you'll deploy a **pre-trained large language model** (Flan-T5) using the SageMaker HuggingFace inference container.

## What You'll Do
1. **Deploy** a Flan-T5 foundation model using the HuggingFace inference container
2. **Experiment** with different prompt strategies for 4 NLP tasks
3. **Compare** results across prompt formulations
4. **Write** a 300-word analysis of your findings

## Important
- **Do NOT change instance types** — they are set for Learner Lab limits
- You should have the Part 1 XGBoost endpoint still running
- You will now have **TWO endpoints** running — complete Part 2 in one sitting if possible


In [ ]:
import sagemaker
import boto3
import pandas as pd
import json
from sagemaker import get_execution_role
from sagemaker.huggingface import HuggingFaceModel

role = get_execution_role()
session = sagemaker.Session()
region = boto3.Session().region_name
print(f"Region: {region}")


## Phase 2A: Deploy Flan-T5 with the HuggingFace Inference Container

The **SageMaker HuggingFace inference container** lets you deploy any HuggingFace model directly to a SageMaker endpoint. Instead of training from scratch (which requires massive datasets and GPU clusters), we can deploy a ready-to-use model in minutes.

**Flan-T5 Small** is a 77M parameter text-to-text model trained by Google. It can:
- Classify text (sentiment, topic)
- Summarize text
- Answer questions
- Extract information
- Generate text based on instructions

It runs on CPU instances (no GPU required), making it perfect for our Learner Lab environment. We use the HuggingFace container because it supports CPU instances directly — the model is downloaded from the HuggingFace Hub at deployment time.


In [ ]:
# Deploy Flan-T5 Small using the HuggingFace inference container
# DO NOT change the model ID or instance_type

# Retrieve the HuggingFace inference container image
image_uri = sagemaker.image_uris.retrieve(
    framework='huggingface',
    region=region,
    version='4.37.0',
    py_version='py310',
    image_scope='inference',
    base_framework_version='pytorch2.1.0',
    instance_type='ml.m5.large',
)

model = HuggingFaceModel(
    image_uri=image_uri,
    env={
        'HF_MODEL_ID': 'google/flan-t5-small',
        'HF_TASK': 'text2text-generation'
    },
    role=role,
    sagemaker_session=session,
)

print("Deploying Flan-T5 Small... (this will take ~5-8 minutes)")
predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',  # DO NOT CHANGE — Learner Lab restriction
    # Fallback: if ml.m5.large fails, try ml.c5.xlarge instead
)
print(f"Endpoint deployed: {predictor.endpoint_name}")
print("\n📸 SCREENSHOT: Open SageMaker Console > Endpoints and take a screenshot showing this endpoint InService.")


In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

def ask_flan(prompt, max_length=200):
    """Send a prompt to Flan-T5 and return the generated text.

    Args:
        prompt: The instruction/question to send to the model
        max_length: Maximum tokens in the response (default 200)

    Returns:
        Generated text string
    """
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_length": max_length,
            "temperature": 0.3,
            "do_sample": True,
        }
    }
    response = predictor.predict(payload)
    # HuggingFace container returns [{'generated_text': '...'}]
    return response[0]['generated_text']

# Quick test
result = ask_flan("Translate to French: The restaurant was excellent.")
print(f"Test response: {result}")


### What Are Foundation Models?

| Traditional ML | Foundation Models |
|---------------|-------------------|
| Train on YOUR data for ONE task | Pre-trained on massive datasets for MANY tasks |
| Small models (MB) | Large models (GB-TB) |
| Weeks to train | Minutes to deploy (pre-trained) |
| Task-specific | General-purpose (controlled by prompts) |
| Need labeled data | Work with natural language instructions |

**How we deploy foundation models on SageMaker:**

AWS offers several ways to use foundation models:
- **Bedrock** — Fully managed API (like calling an API). You don't manage the infrastructure.
- **JumpStart** — AWS-curated model catalog deployed to your SageMaker endpoint.
- **HuggingFace container** — Deploy any model from the HuggingFace Hub to your SageMaker endpoint. You specify the model ID and the container handles downloading and serving.

In Learner Lab, we use the **HuggingFace inference container** because it works on CPU instances (`ml.m5.large`) that are available in our environment. The container downloads the model from HuggingFace Hub at deployment time.


In [ ]:
# Load the Charlotte reviews dataset
# (If files are not in the current directory, download them first)
import os
if not os.path.exists('charlotte_reviews.csv'):
    !aws s3 cp s3://dsba6190-smith-lab3/data/charlotte_restaurants.csv ./
    !aws s3 cp s3://dsba6190-smith-lab3/data/charlotte_reviews.csv ./

reviews_df = pd.read_csv('charlotte_reviews.csv')
restaurants_df = pd.read_csv('charlotte_restaurants.csv')

# Merge to get restaurant details with each review
reviews_full = reviews_df.merge(
    restaurants_df[['restaurant_id', 'name', 'cuisine', 'neighborhood', 'avg_rating']],
    on='restaurant_id'
)

print(f"Loaded {len(reviews_full)} reviews across {reviews_full['name'].nunique()} restaurants")
print(f"\nSample review:")
sample = reviews_full.iloc[0]
print(f"  Restaurant: {sample['name']} ({sample['cuisine']}, {sample['neighborhood']})")
print(f"  Rating: {sample['rating']}/5")
print(f"  Review: {sample['review_text']}")


## Phase 2B: Prompt Engineering Exercises

### Task 1: Sentiment Classification

Can Flan-T5 correctly classify review sentiment? Try **three different prompt strategies** and compare their accuracy.

**Your goal:** Classify each review as **Positive**, **Negative**, or **Mixed**.

**Evaluation:** You'll compare the model's classification against the actual star rating:
- 4-5 stars → Positive
- 1-2 stars → Negative
- 3 stars → Mixed


In [ ]:
# Select 10 test reviews with known ratings (stratified across rating levels)
test_reviews = pd.concat([
    reviews_full[reviews_full['rating'] == 1].sample(2, random_state=42),
    reviews_full[reviews_full['rating'] == 2].sample(2, random_state=42),
    reviews_full[reviews_full['rating'] == 3].sample(2, random_state=42),
    reviews_full[reviews_full['rating'] == 4].sample(2, random_state=42),
    reviews_full[reviews_full['rating'] == 5].sample(2, random_state=42),
]).reset_index(drop=True)

# Map ratings to expected sentiment
def rating_to_sentiment(r):
    if r >= 4: return "Positive"
    elif r <= 2: return "Negative"
    else: return "Mixed"

test_reviews['expected_sentiment'] = test_reviews['rating'].apply(rating_to_sentiment)
print("Test reviews selected:")
for i, row in test_reviews.iterrows():
    print(f"  [{i}] Rating: {row['rating']} | Expected: {row['expected_sentiment']} | {row['review_text'][:80]}...")


In [ ]:
# TODO: Zero-shot sentiment classification
# =============================================
# Write a prompt that asks Flan-T5 to classify sentiment with NO examples.
# Run it on all 10 test reviews and record the results.

results_zeroshot = []
for i, row in test_reviews.iterrows():
    prompt = ""  # TODO: Write your zero-shot prompt here
    # Example structure: f"Classify the sentiment of this review as Positive, Negative, or Mixed: {row['review_text']}"

    response = ask_flan(prompt)
    results_zeroshot.append(response.strip())
    print(f"[{i}] Expected: {row['expected_sentiment']:8s} | Got: {response.strip()}")

# Calculate accuracy
correct = sum(1 for exp, got in zip(test_reviews['expected_sentiment'], results_zeroshot)
              if exp.lower() in got.lower())
print(f"\nZero-shot accuracy: {correct}/10 ({100*correct/10:.0f}%)")
# =============================================


In [ ]:
# TODO: Few-shot sentiment classification
# =============================================
# Write a prompt that includes 2-3 EXAMPLES before the actual review.
# The examples teach the model what you expect.

results_fewshot = []
for i, row in test_reviews.iterrows():
    prompt = ""  # TODO: Write your few-shot prompt here
    # Example structure:
    # f"""Classify the sentiment as Positive, Negative, or Mixed.
    #
    # Review: "The food was amazing and the service was great!"
    # Sentiment: Positive
    #
    # Review: "Terrible experience, cold food and rude staff."
    # Sentiment: Negative
    #
    # Review: "{row['review_text']}"
    # Sentiment:"""

    response = ask_flan(prompt)
    results_fewshot.append(response.strip())
    print(f"[{i}] Expected: {row['expected_sentiment']:8s} | Got: {response.strip()}")

correct = sum(1 for exp, got in zip(test_reviews['expected_sentiment'], results_fewshot)
              if exp.lower() in got.lower())
print(f"\nFew-shot accuracy: {correct}/10 ({100*correct/10:.0f}%)")
# =============================================


In [ ]:
# TODO: Chain-of-thought sentiment classification
# =============================================
# Write a prompt that asks the model to EXPLAIN its reasoning before classifying.
# This often improves accuracy on nuanced cases.

results_cot = []
for i, row in test_reviews.iterrows():
    prompt = ""  # TODO: Write your chain-of-thought prompt here
    # Example: "Read this review and first list the positive and negative points,
    #           then classify the overall sentiment as Positive, Negative, or Mixed."

    response = ask_flan(prompt, max_length=300)
    results_cot.append(response.strip())
    print(f"[{i}] Expected: {row['expected_sentiment']:8s} | Response: {response.strip()[:100]}")

# Note: Accuracy is harder to compute for chain-of-thought since the answer
# is embedded in the explanation. Manually review the outputs.
print("\nReview the responses above. Does chain-of-thought improve classification?")
# =============================================


**Record your observations for Task 1:**
- Which prompt strategy gave the best accuracy?
- Where did each strategy struggle?
- Were there specific types of reviews that were harder to classify?


### Task 2: Review Summarization

Given multiple reviews for the same restaurant, can Flan-T5 generate a useful one-paragraph summary?


In [ ]:
# Pick a restaurant with several reviews
restaurant_name = reviews_full['name'].value_counts().index[0]  # Most-reviewed restaurant
restaurant_reviews = reviews_full[reviews_full['name'] == restaurant_name].head(5)

print(f"Restaurant: {restaurant_name}")
print(f"Reviews ({len(restaurant_reviews)}):\n")
combined_reviews = ""
for i, row in restaurant_reviews.iterrows():
    print(f"  [{row['rating']}/5]: {row['review_text']}")
    print()
    combined_reviews += f"Review: {row['review_text']}\n"


In [ ]:
# TODO: Summarization — Prompt Variation 1
# =============================================
# Write a direct summarization prompt

prompt_1 = ""  # TODO: e.g., f"Summarize these restaurant reviews:\n{combined_reviews}"
summary_1 = ask_flan(prompt_1, max_length=200)
print("Prompt 1 (direct):")
print(summary_1)
# =============================================


In [ ]:
# TODO: Summarization — Prompt Variation 2
# =============================================
# Write a role-based prompt (e.g., "As a food critic...")

prompt_2 = ""  # TODO: Try a different approach
summary_2 = ask_flan(prompt_2, max_length=200)
print("Prompt 2 (role-based):")
print(summary_2)
# =============================================


In [ ]:
# TODO: Summarization — Prompt Variation 3 (optional)
# =============================================
# Try a structured prompt (e.g., "Summarize the food quality, service, and atmosphere...")

prompt_3 = ""  # TODO
summary_3 = ask_flan(prompt_3, max_length=200)
print("Prompt 3 (structured):")
print(summary_3)
# =============================================


### Task 3: Information Extraction

Can Flan-T5 extract structured information from unstructured review text? Try extracting:
- Specific **dishes mentioned**
- **Service quality** descriptors
- **Atmosphere** descriptors


In [ ]:
# Pick a detailed review for extraction
detailed_review = reviews_full.loc[reviews_full['review_text'].str.len().idxmax()]
print(f"Restaurant: {detailed_review['name']}")
print(f"Review: {detailed_review['review_text']}")


In [ ]:
# TODO: Information Extraction — Free-form
# =============================================
prompt_freeform = ""  # TODO: e.g., f"Extract the dishes, service quality, and atmosphere from this review: {detailed_review['review_text']}"
result_freeform = ask_flan(prompt_freeform, max_length=200)
print("Free-form extraction:")
print(result_freeform)
# =============================================


In [ ]:
# TODO: Information Extraction — Structured output
# =============================================
prompt_structured = ""  # TODO: Ask for JSON-like or labeled output
# e.g., "From this review, list: Dishes: ..., Service: ..., Atmosphere: ..."
result_structured = ask_flan(prompt_structured, max_length=200)
print("Structured extraction:")
print(result_structured)
# =============================================


### Task 4: Restaurant Profile Generation

Combine structured data (from Part 1's dataset) with review text to generate a 3-sentence "restaurant profile" suitable for a food guide. Compare profiles for a well-reviewed vs. poorly-reviewed restaurant.


In [ ]:
# Get a high-rated and a low-rated restaurant
high_rated = restaurants_df.loc[restaurants_df['avg_rating'].idxmax()]
low_rated = restaurants_df.loc[restaurants_df['avg_rating'].idxmin()]

high_reviews = reviews_full[reviews_full['restaurant_id'] == high_rated['restaurant_id']].head(3)
low_reviews = reviews_full[reviews_full['restaurant_id'] == low_rated['restaurant_id']].head(3)

print(f"High-rated: {high_rated['name']} ({high_rated['avg_rating']}/5, {high_rated['cuisine']}, {high_rated['neighborhood']})")
print(f"Low-rated: {low_rated['name']} ({low_rated['avg_rating']}/5, {low_rated['cuisine']}, {low_rated['neighborhood']})")


In [ ]:
# TODO: Generate restaurant profiles
# =============================================
# Combine structured data + reviews into a prompt that generates a 3-sentence profile

# Profile for high-rated restaurant
high_review_text = "\n".join(high_reviews['review_text'].tolist())
prompt_high = ""  # TODO: Write your profile generation prompt
profile_high = ask_flan(prompt_high, max_length=200)
print(f"Profile — {high_rated['name']}:")
print(profile_high)
print()

# Profile for low-rated restaurant
low_review_text = "\n".join(low_reviews['review_text'].tolist())
prompt_low = ""  # TODO: Write your profile generation prompt
profile_low = ask_flan(prompt_low, max_length=200)
print(f"Profile — {low_rated['name']}:")
print(profile_low)
# =============================================


## Phase 2C: Analysis Write-up

**TODO:** Write a minimum **300-word analysis** comparing your prompt engineering experiments. Address these questions:

1. **Which prompt formulation worked best for each task and why?** Be specific — reference actual outputs.
2. **Where did Flan-T5 Small struggle?** What types of tasks or inputs caused poor results?
3. **What might improve with a larger model?** (Flan-T5 Small is 77M parameters; Flan-T5 Large is 783M; GPT-4 is ~1.7T)
4. **How could these GenAI capabilities integrate with the ML model from Part 1?** Think about a production system that uses both the rating predictor and the text analyzer.

*Write your analysis in the cell below:*


**Your Analysis (minimum 300 words):**

*(TODO: Write your analysis here)*



---

**You now have TWO running endpoints.** Proceed immediately to **Part 3: Cleanup** to delete both endpoints and avoid unnecessary charges.
